# 05 — Detecção de anomalias por fornecedor (Isolation Forest)

Perfil comportamental de 38.641 fornecedores (≥5 licitações) com features de
comportamento (taxa de vitória, % dispensa/inexigibilidade, concentração por UG),
normalizadas com StandardScaler. Isolation Forest, contaminação 2%.

## Achados
- **1ª versão (features de volume):** modelo pegou os maiores fornecedores nacionais
  como "anômalos" — volume ≠ suspeita. Corrigido normalizando escalas e usando
  features comportamentais.
- **2ª versão:** topo dominado por fabricantes de equipamento científico (Bruker,
  Shimadzu, Thermo Fisher) com ~100% vitória e alta inexigibilidade — anomalia
  estatística LEGÍTIMA (fornecedor exclusivo). Falso positivo explicável.
- **Alvo real destacado:** W ENGENHARIA (1.619 licitações, 100% vitória, 100%
  dispensa) — padrão sem justificativa de exclusividade, recomendado para auditoria.

## Validação contra CEIS
- Taxa base de sancionados: 7,48%. Nos top 500 anômalos: 0,20%. Lift: 0,03x.
- **Interpretação:** o modelo detecta anomalia de *dominância de mercado* (players
  grandes e exclusivos), não *fraude de idoneidade* (empresas-fantasma, laranjas),
  que é o que o CEIS registra. São fenômenos distintos. O lift baixo confirma essa
  separação — não invalida o modelo, delimita seu escopo.

In [7]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np

# Dataset limpo da fase 2 (sem placeholders)
df = pd.read_parquet("../data/processed/licitacoes_limpo.parquet")
print(df.shape)
df.columns.tolist()

(84262, 18)


['Número Licitação',
 'Código UG',
 'Nome UG',
 'Código Modalidade Compra',
 'Modalidade Compra',
 'Número Processo',
 'Objeto',
 'Situação Licitação',
 'Código Órgão Superior',
 'Nome Órgão Superior',
 'Código Órgão',
 'Nome Órgão',
 'UF',
 'Município',
 'Data Resultado Compra',
 'Data Abertura',
 'Valor Licitação',
 'arquivo_origem']

In [8]:
import sys
sys.path.append("..")
import pandas as pd
import numpy as np

df_lic = pd.read_parquet("../data/processed/licitacoes_limpo.parquet")
print("Licitações limpas:", df_lic.shape)

Licitações limpas: (84262, 18)


In [9]:
from src.carga import carregar_participantes

part = carregar_participantes("../data/raw")
print("Participantes:", part.shape, "| Fornecedores:", part["Código Participante"].nunique())

c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns (0: Código Participante) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(arq, sep=";", encoding="latin-1", decimal=",")
c:\auditoria-ml\notebooks\..\src\carga.py:30: DtypeWarning: Columns 

Participantes: (4607976, 14) | Fornecedores: 81766


In [10]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [11]:
from sklearn.ensemble import IsolationForest

# --- Feature engineering por fornecedor ---
g = part.groupby("Código Participante")
feat = pd.DataFrame({
    "n_licitacoes": g.size(),
    "n_vitorias": g["Flag Vencedor"].apply(lambda x: (x == "SIM").sum()),
    "n_ugs": g["Código UG"].nunique(),
    "n_modalidades": g["Modalidade Compra"].nunique(),
    "pct_dispensa": g["Modalidade Compra"].apply(lambda x: (x == "Dispensa de Licitação").mean()),
    "pct_inexig": g["Modalidade Compra"].apply(lambda x: (x == "Inexigibilidade de Licitação").mean()),
})
feat["taxa_vitoria"] = feat["n_vitorias"] / feat["n_licitacoes"]

# So fornecedores com atividade minima (senao vira ruido)
feat = feat[feat["n_licitacoes"] >= 5].copy()
print("Fornecedores analisados:", len(feat))

# --- Isolation Forest ---
cols = ["n_licitacoes", "n_vitorias", "n_ugs", "n_modalidades",
        "pct_dispensa", "pct_inexig", "taxa_vitoria"]
X = feat[cols].fillna(0)

iso = IsolationForest(contamination=0.02, random_state=42)
feat["anomaly_score"] = iso.fit_predict(X)
feat["score"] = iso.score_samples(X)  # quanto menor, mais anomalo

# --- Ranking dos mais anomalos ---
nomes = part.drop_duplicates("Código Participante").set_index("Código Participante")["Nome Participante"]
anomalos = feat.sort_values("score").head(20).copy()
anomalos["nome"] = anomalos.index.map(nomes)
anomalos[["nome", "n_licitacoes", "taxa_vitoria", "pct_dispensa", "pct_inexig", "n_ugs", "score"]]

KeyboardInterrupt: 

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

g = part.groupby("Código Participante")
feat = pd.DataFrame({
    "n_licitacoes": g.size(),
    "n_vitorias": g["Flag Vencedor"].apply(lambda x: (x == "SIM").sum()),
    "n_ugs": g["Código UG"].nunique(),
    "n_modalidades": g["Modalidade Compra"].nunique(),
    "pct_dispensa": g["Modalidade Compra"].apply(lambda x: (x == "Dispensa de Licitação").mean()),
    "pct_inexig": g["Modalidade Compra"].apply(lambda x: (x == "Inexigibilidade de Licitação").mean()),
})
feat["taxa_vitoria"] = feat["n_vitorias"] / feat["n_licitacoes"]
# Concentracao: licitacoes por UG (quem opera muito em poucos orgaos = suspeito)
feat["lic_por_ug"] = feat["n_licitacoes"] / feat["n_ugs"]

feat = feat[feat["n_licitacoes"] >= 5].copy()
print("Fornecedores analisados:", len(feat))

# Features COMPORTAMENTAIS (proporcoes + concentracao), sem volume bruto dominante
cols = ["taxa_vitoria", "pct_dispensa", "pct_inexig", "lic_por_ug", "n_modalidades"]
X = StandardScaler().fit_transform(feat[cols].fillna(0))

iso = IsolationForest(contamination=0.02, random_state=42)
iso.fit(X)
feat["score"] = iso.score_samples(X)

nomes = part.drop_duplicates("Código Participante").set_index("Código Participante")["Nome Participante"]
anomalos = feat.sort_values("score").head(20).copy()
anomalos["nome"] = anomalos.index.map(nomes)
anomalos[["nome", "n_licitacoes", "taxa_vitoria", "pct_dispensa", "pct_inexig", "lic_por_ug", "score"]]

KeyboardInterrupt: 

In [ ]:
import requests, io, zipfile

url = "https://portaldatransparencia.gov.br/download-de-dados/ceis/20240725"  # ajusta a data se precisar
# Se a URL falhar, baixa manual de: portaldatransparencia.gov.br/download-de-dados/ceis

In [ ]:
# Carrega CEIS
ceis = pd.read_csv(
    "../data/raw/20260724_CEIS.csv",
    sep=";", encoding="latin-1"
)
print("Colunas CEIS:", ceis.columns.tolist())
print("Registros:", len(ceis))
ceis.head(3)

Colunas CEIS: ['CADASTRO', 'CÓDIGO DA SANÇÃO', 'TIPO DE PESSOA', 'CPF OU CNPJ DO SANCIONADO', 'NOME DO SANCIONADO', 'NOME INFORMADO PELO ÓRGÃO SANCIONADOR', 'RAZÃO SOCIAL - CADASTRO RECEITA', 'NOME FANTASIA - CADASTRO RECEITA', 'NÚMERO DO PROCESSO', 'CATEGORIA DA SANÇÃO', 'DATA INÍCIO SANÇÃO', 'DATA FINAL SANÇÃO', 'DATA PUBLICAÇÃO', 'PUBLICAÇÃO', 'DETALHAMENTO DO MEIO DE PUBLICAÇÃO', 'DATA DO TRÂNSITO EM JULGADO', 'ABRAGÊNCIA DA SANÇÃO', 'ÓRGÃO SANCIONADOR', 'UF ÓRGÃO SANCIONADOR', 'ESFERA ÓRGÃO SANCIONADOR', 'FUNDAMENTAÇÃO LEGAL', 'DATA ORIGEM INFORMAÇÃO', 'ORIGEM INFORMAÇÕES', 'OBSERVAÇÕES']
Registros: 23333


,CADASTRO,CÓDIGO DA SANÇÃO,TIPO DE PESSOA,CPF OU CNPJ DO SANCIONADO,NOME DO SANCIONADO,NOME INFORMADO PELO ÓRGÃO SANCIONADOR,RAZÃO SOCIAL - CADASTRO RECEITA,NOME FANTASIA - CADASTRO RECEITA,NÚMERO DO PROCESSO,CATEGORIA DA SANÇÃO,...,DETALHAMENTO DO MEIO DE PUBLICAÇÃO,DATA DO TRÂNSITO EM JULGADO,ABRAGÊNCIA DA SANÇÃO,ÓRGÃO SANCIONADOR,UF ÓRGÃO SANCIONADOR,ESFERA ÓRGÃO SANCIONADOR,FUNDAMENTAÇÃO LEGAL,DATA ORIGEM INFORMAÇÃO,ORIGEM INFORMAÇÕES,OBSERVAÇÕES
0,CEIS,92390,F,2218151987,VIRGOLINO FRANCISCO VIANA,VIRGOLINO FRANCISCO VIANA,NaN,NaN,00002791620078160132,Impedimento/proibição de contratar com prazo d...,...,NaN,09/09/2016,Sem Informação,Tribunal de Justiça do Estado do Paraná / 1º G...,PR,ESTADUAL,LEI 8429 - ART. 12 - INDEPENDENTEMENTE DAS SAN...,13/03/2018,Conselho Nacional de Justiça (CNJ-DF),NaN
1,CEIS,379678,F,77538650725,ALEXANDRE DE ALBUQUERQUE BRAILE,ALEXANDRE DE ALBUQUERQUE BRAILE,NaN,NaN,00414130420124025101,Impedimento/proibição de contratar com prazo d...,...,NaN,11/11/2024,Sem Informação,1º Grau - TRF2 / Seção Judiciária do Rio de Ja...,RJ,FEDERAL,LEI 8429 - ART. 12 - INDEPENDENTEMENTE DAS SAN...,03/03/2026,Conselho Nacional de Justiça (CNJ-DF),NaN
2,CEIS,89670,F,7097066423,ROBSON ARAUJO DA SILVA,ROBSON ARAúJO DA SILVA,NaN,NaN,08004577520164058402,Impedimento/proibição de contratar com prazo d...,...,NaN,01/09/2017,Sem Informação,1º Grau - TRF5 / Seção Judiciária do Rio Grand...,RN,FEDERAL,LEI 8429 - ART. 12 - INDEPENDENTEMENTE DAS SAN...,15/12/2017,Conselho Nacional de Justiça (CNJ-DF),NaN


In [ ]:
# CNPJs sancionados (so pessoa juridica, tipo J)
ceis_pj = ceis[ceis["TIPO DE PESSOA"] == "J"].copy()
cnpj_sancionados = set(
    ceis_pj["CPF OU CNPJ DO SANCIONADO"]
    .astype(str).str.replace(r"\D", "", regex=True)  # so digitos
)
print("CNPJs sancionados no CEIS:", len(cnpj_sancionados))

# Normaliza os codigos dos fornecedores do mesmo jeito
feat["cnpj_limpo"] = feat.index.astype(str).str.replace(r"\D", "", regex=True)
feat["sancionado"] = feat["cnpj_limpo"].isin(cnpj_sancionados)

# --- VALIDACAO ---
taxa_base = feat["sancionado"].mean()

top_n = 500
top_anomalos = feat.sort_values("score").head(top_n)
taxa_anomalos = top_anomalos["sancionado"].mean()

print(f"\nTaxa base de sancionados (todos {len(feat)} fornecedores): {taxa_base:.2%}")
print(f"Taxa de sancionados nos top {top_n} anômalos: {taxa_anomalos:.2%}")
print(f"Lift: {taxa_anomalos / taxa_base:.2f}x")

CNPJs sancionados no CEIS: 9407

Taxa base de sancionados (todos 38641 fornecedores): 7.48%
Taxa de sancionados nos top 500 anômalos: 0.20%
Lift: 0.03x
